<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/process_automation_api_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Process Automation API Demo

Use NeqSim's string-addressable automation API to inspect a small process, read values, change setpoints, and export structured snapshots.


## Setup

Install and import NeqSim classes used by the demo.


In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
from neqsim import jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem


## Build a Small Process

The flowsheet contains a feed stream, a heater/cooler, a separator, and a compressor.


In [ ]:
def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid

fluid = make_gas()
feed = Stream("Feed gas", fluid)
feed.setFlowRate(10000.0, "kg/hr")

Heater = jneqsim.process.equipment.heatexchanger.Heater
Separator = jneqsim.process.equipment.separator.Separator
Compressor = jneqsim.process.equipment.compressor.Compressor

cooler = Heater("Gas cooler", feed)
cooler.setOutTemperature(273.15 + 15.0)
separator = Separator("Inlet separator", cooler.getOutStream())
compressor = Compressor("Export compressor", separator.getGasOutStream())
compressor.setOutletPressure(90.0)

process = ProcessSystem()
for unit in [feed, cooler, separator, compressor]:
    process.add(unit)
process.run()


## Discover Units and Variables

The automation facade exposes units and variables without navigating Java class internals.


In [ ]:
auto = process.getAutomation()
print(list(auto.getUnitList()))

for variable in auto.getVariableList("Export compressor"):
    address = variable.getAddress() if hasattr(variable, "getAddress") else str(variable)
    variable_type = variable.getVariableType() if hasattr(variable, "getVariableType") else ""
    unit = ""
    if hasattr(variable, "getUnit"):
        unit = variable.getUnit()
    elif hasattr(variable, "getDefaultUnit"):
        unit = variable.getDefaultUnit()
    elif hasattr(variable, "getDisplayUnit"):
        unit = variable.getDisplayUnit()
    print(address, unit, variable_type)

## Read and Change Setpoints

Read compressor power, then change the outlet pressure and run only when the model is dirty.


In [ ]:
power1 = auto.getVariableValue("Export compressor.power", "kW")
auto.setVariableValue("Export compressor.outletPressure", 110.0, "bara")
auto.runIfDirty()
power2 = auto.getVariableValue("Export compressor.power", "kW")
pd.DataFrame([{"case": "base", "power_kW": power1}, {"case": "higher pressure", "power_kW": power2}])


## Structured Snapshots

Snapshots and topology are JSON strings that can be stored, compared, or used by agents.


In [ ]:
snapshot = json.loads(str(auto.snapshot("*")))
topology = json.loads(str(auto.getTopology()))
print(json.dumps({"snapshot_schema": snapshot.get("schemaVersion"), "topology_keys": list(topology.keys())}, indent=2))
